In [1]:
# get base model
# create train process
# check training parameters
# run train process

In [2]:
import sys
sys.path.append("../")

from src.ner.loading import Loader
from pathlib import Path
from src.ner.models import Model
from src.ner.processing import Processor, label2id, label_list, id2label
from src.ner.training import TrainingProcess

/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# get base model
model = Model(
    name="microsoft/deberta-v3-base",
    label_list=label_list,
    id2label=id2label,
    label2id=label2id
)

The tokenizer you are loading from '/Users/ayeustsihneyeu/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [4]:
loader = Loader(path=Path("../data/entities.json"), test_k=0.2, seed=42)
ds = loader.load()
train_ds, test_ds = loader.train_test_split(dataset=ds)
train_ds, val_ds = loader.train_test_split(dataset=train_ds)
(len(ds), len(train_ds), len(val_ds), len(test_ds))

(100, 64, 16, 20)

In [5]:
train_ds

Dataset({
    features: ['text', 'entities'],
    num_rows: 64
})

In [6]:
# preprocess datasets
preprocessor = Processor(
    tokenizer=model.tokenizer,
)

preprocess_func = preprocessor.preprocess

train_ds = train_ds.map(lambda x: preprocess_func(x))
val_ds = val_ds.map(lambda x: preprocess_func(x))

Map: 100%|██████████| 16/16 [00:00<00:00, 2648.23 examples/s]


In [7]:
# create train process
training_process = TrainingProcess(
    model=model,
    train_ds=train_ds,
    val_ds=val_ds,
    output_dir="../ner_model",
    r=8
)

In [8]:
# create peft model
peft_model = training_process._peft_model()
peft_model

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 55777.57it/s]
DebertaV2ForTokenClassification LOAD REPORT from: /Users/ayeustsihneyeu/.cache/huggingface/hub/models--microsoft--deberta-v3-base/snapshots/8ccc9b6f36199bec6961081d44eb72fb3f7353f3
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.bias      

PeftModelForTokenClassification(
  (base_model): LoraModel(
    (model): DebertaV2ForTokenClassification(
      (deberta): DebertaV2Model(
        (embeddings): DebertaV2Embeddings(
          (word_embeddings): Embedding(128100, 768, padding_idx=0)
          (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): DebertaV2Encoder(
          (layer): ModuleList(
            (0-11): 12 x DebertaV2Layer(
              (attention): DebertaV2Attention(
                (self): DisentangledSelfAttention(
                  (query_proj): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=False)
              

In [9]:
# check training parameters
peft_model.print_trainable_parameters()

trainable params: 498,505 || all params: 184,386,194 || trainable%: 0.2704


In [10]:
# get train args
training_process._training_args()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [11]:
# get trainer
training_process._trainer()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
# run train process
training_process.train_save()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,102.671875,21.843750,0.071749,0.044444,0.186047
2,145.535156,26.781250,0.130719,0.090909,0.232558
3,45.546875,15.445312,0.290076,0.215909,0.441860
4,37.003906,12.468750,0.379310,0.301370,0.511628
5,28.703125,14.351562,0.251852,0.184783,0.395349
6,18.540039,9.984375,0.450450,0.367647,0.581395
7,20.659180,9.640625,0.500000,0.426230,0.604651
8,6.798828,10.539062,0.482143,0.391304,0.627907
9,23.791016,9.109375,0.563107,0.483333,0.674419
10,7.936035,8.281250,0.604167,0.547170,0.674419


/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/ayeustsihneyeu/PythonProjects/mask_sft/.masksft/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinn